<a href="https://colab.research.google.com/github/vcasadei/Redes-Neurais-CESAR-School/blob/2026.01-A/4%20-%20Redes%20Neurais/4.4_otimizadores_regressao_linear.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# 4.4 — Otimizadores do gradiente descendente na regressão linear

Até aqui todos os treinamentos usaram o mesmo otimizador: o gradiente descendente estocástico (SGD), que dá um passo na direção oposta ao gradiente. Existem variações desse passo que usam a história dos gradientes anteriores. Neste notebook voltamos ao problema do notebook 2.1 — prever o comprimento da pétala da flor Íris versicolor a partir do comprimento da sépala — e trocamos apenas o otimizador. Como o modelo tem só dois parâmetros, dá para ver o caminho de cada otimizador no mapa da função de custo.

## Objetivos

- Conhecer as regras de atualização do SGD, do SGD com momentum, do momentum de Nesterov, do RMSprop e do Adam, e o que cada símbolo representa.
- Ver, no plano dos parâmetros $w_0 \times w_1$, o caminho que cada otimizador percorre até o mínimo do custo.
- Comparar os otimizadores com os mesmos dados, o mesmo ponto inicial, a mesma taxa de aprendizado e o mesmo número de épocas.
- Verificar como a taxa de aprendizado e o tamanho do mini-batch mudam essa comparação.

Referências:
- S. Ruder, *An overview of gradient descent optimization algorithms* (2016): https://www.ruder.io/optimizing-gradient-descent/
- Documentação dos otimizadores do PyTorch (as fórmulas abaixo seguem a implementação de `torch.optim`): https://pytorch.org/docs/stable/optim.html
- O código de visualização foi inspirado nos tutoriais de Morvan Zhou: https://mofanpy.com/tutorials/

## Métodos de otimização do gradiente descendente

Notação usada em todas as fórmulas:

- $W_t$: vetor de parâmetros no passo $t$. Aqui $W = [w_0 \ \ w_1]$, com $w_0$ o *bias* e $w_1$ o peso da reta $\hat{y} = w_0 + w_1 x$ (nos slides, $w_0 = \theta_0$ e $w_1 = \theta_1$).
- $t$: o passo (iteração), isto é, uma atualização dos parâmetros. Uma época tem tantos passos quantos mini-batches.
- $J(W)$: função de custo (loss), aqui o erro quadrático médio (MSE).
- $g_t = \nabla J(W_t)$: gradiente da função de custo em relação aos parâmetros, calculado no mini-batch do passo $t$.
- $\eta$: taxa de aprendizado (learning rate), o argumento `lr` dos otimizadores.

### Gradiente descendente (SGD)

$$ W_{t+1} = W_t - \eta \, g_t $$

O passo é proporcional ao gradiente atual e não usa nenhuma informação dos passos anteriores. No PyTorch: `torch.optim.SGD(params, lr=lr)`.

### Gradiente descendente com momentum

$$ v_{t+1} = \gamma \, v_t + g_t $$
$$ W_{t+1} = W_t - \eta \, v_{t+1} $$

- $v_t$: velocidade, um acúmulo dos gradientes anteriores, com $v_0 = 0$.
- $\gamma$: coeficiente de momentum, entre 0 e 1 (argumento `momentum`; aqui 0,9). Com $\gamma = 0$ voltamos ao SGD.

Quando gradientes sucessivos apontam na mesma direção, $v$ cresce e o passo aumenta; quando o sinal se alterna (oscilação de um lado para outro de um vale), os termos se cancelam em parte. No PyTorch: `torch.optim.SGD(params, lr=lr, momentum=0.9)`, que implementa exatamente as duas equações acima.

### Momentum de Nesterov

A ideia de Nesterov é calcular o gradiente no ponto para onde o momentum vai levar os parâmetros, $W_t - \eta\gamma v_t$, em vez de no ponto atual:

$$ v_{t+1} = \gamma \, v_t + \nabla J(W_t - \eta \gamma \, v_t) $$
$$ W_{t+1} = W_t - \eta \, v_{t+1} $$

Os símbolos são os mesmos do momentum. O PyTorch usa uma forma reescrita, equivalente após uma mudança de variável, que só precisa do gradiente no ponto atual $g_t$:

$$ v_{t+1} = \gamma \, v_t + g_t $$
$$ W_{t+1} = W_t - \eta \, (g_t + \gamma \, v_{t+1}) $$

No PyTorch: `torch.optim.SGD(params, lr=lr, momentum=0.9, nesterov=True)`.

### Métodos com taxa de aprendizado adaptativa para cada parâmetro

Nos métodos acima, todos os parâmetros usam o mesmo $\eta$. Os métodos a seguir dividem o passo de cada parâmetro $i$ por uma medida do tamanho dos seus gradientes recentes: parâmetros com gradientes grandes dão passos menores e vice-versa. As operações são feitas elemento a elemento ($g_{t,i}$ é a componente $i$ do gradiente) e $\epsilon$ é uma constante pequena que evita divisão por zero (argumento `eps`).

**Adagrad** — $G_{t,i}$ é a soma dos quadrados de todos os gradientes do parâmetro $i$ desde o início do treinamento:

$$ G_{t,i} = \sum_{\tau=1}^{t} g_{\tau,i}^2 \qquad\qquad W_{t+1,i} = W_{t,i} - \frac{\eta}{\sqrt{G_{t,i}} + \epsilon} \, g_{t,i} $$

Como $G$ só cresce, o passo efetivo diminui ao longo de todo o treinamento. O Adagrad não é usado neste notebook; ele aparece aqui porque o RMSprop corrige esse problema.

**RMSprop** — troca a soma por uma média móvel exponencial $s_{t,i}$ dos quadrados dos gradientes, que "esquece" os gradientes antigos:

$$ s_{t,i} = \alpha \, s_{t-1,i} + (1 - \alpha) \, g_{t,i}^2 \qquad\qquad W_{t+1,i} = W_{t,i} - \frac{\eta}{\sqrt{s_{t,i}} + \epsilon} \, g_{t,i} $$

- $\alpha$: fator de decaimento da média móvel (argumento `alpha`; aqui 0,9, o padrão do PyTorch é 0,99), com $s_0 = 0$.

No PyTorch: `torch.optim.RMSprop(params, lr=lr, alpha=0.9)`. O Adadelta (usado no notebook 4.6) é uma variante do RMSprop que também mantém uma média móvel dos quadrados das próprias atualizações.

**Adam** — combina as duas ideias: uma média móvel dos gradientes $m_t$ (como o momentum) e uma média móvel dos quadrados $s_t$ (como o RMSprop):

$$ m_t = \beta_1 \, m_{t-1} + (1 - \beta_1) \, g_t \qquad\qquad s_t = \beta_2 \, s_{t-1} + (1 - \beta_2) \, g_t^2 $$
$$ \hat{m}_t = \frac{m_t}{1 - \beta_1^t} \qquad\qquad \hat{s}_t = \frac{s_t}{1 - \beta_2^t} \qquad\qquad W_{t+1} = W_t - \frac{\eta}{\sqrt{\hat{s}_t} + \epsilon} \, \hat{m}_t $$

- $\beta_1$, $\beta_2$: fatores de decaimento das duas médias (argumento `betas`; aqui $(0{,}9;\ 0{,}99)$, o padrão é $(0{,}9;\ 0{,}999)$), com $m_0 = s_0 = 0$.
- $\hat{m}_t$, $\hat{s}_t$: médias com correção de viés. Como as médias começam em zero, nos primeiros passos elas ficam pequenas demais; dividir por $1 - \beta^t$ compensa isso ($\beta^t$ é $\beta$ elevado ao número do passo).

No PyTorch: `torch.optim.Adam(params, lr=lr, betas=(0.9, 0.99))`.

### Ilustração de vários métodos

As animações abaixo são de Alec Radford e aparecem no artigo de Ruder citado acima. Elas **não** usam os dados deste notebook: mostram o caminho de seis otimizadores (SGD, Momentum, NAG = Nesterov, Adagrad, Adadelta e RMSprop; o Adam não está incluído) numa função de teste com duas variáveis. Na primeira, as curvas de nível são da função de Beale e a estrela marca o mínimo; na segunda, os otimizadores partem de um ponto de sela (um ponto onde a função sobe numa direção e desce na outra).

Observe que os caminhos são diferentes: alguns ultrapassam o mínimo e voltam, outros vão mais direto. Qual deles é melhor depende da função e dos hiperparâmetros; é isso que vamos verificar no nosso problema.

![Otimizadores nas curvas de nível da função de Beale](https://raw.githubusercontent.com/vcasadei/images/master/contours_evaluation_optimizers.gif)
![Otimizadores num ponto de sela](https://raw.githubusercontent.com/vcasadei/images/master/saddle_point_evaluation_optimizers.gif)

## Importação das bibliotecas

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython import display
from sklearn.datasets import load_iris
from torch.utils.data import TensorDataset, DataLoader

np.random.seed(1234)
torch.manual_seed(1234);

## Dados: flores Íris (versicolor)

Os mesmos dados do notebook 2.1: as 50 amostras da variedade versicolor, com o comprimento das sépalas como entrada $x$ e o comprimento das pétalas como saída $y$, normalizados para o intervalo $[0, 1]$ (normalização min-max).

In [ ]:
iris = load_iris()
data = iris.data[iris.target == 1, ::2]  # versicolor; colunas 0 e 2: comprimento das sépalas e das pétalas

x_in = data[:, 0:1]   # shape (50, 1)
y_in = data[:, 1:2]   # shape (50, 1)
print('x_in:', x_in.shape, ' y_in:', y_in.shape)

In [ ]:
x = x_in - x_in.min()
x /= x.max()    # normalização: após subtrair o mínimo, o máximo vale max - min
y = y_in - y_in.min()
y /= y.max()

plt.scatter(x, y)
plt.xlabel('Comprimento normalizado das sépalas')
plt.ylabel('Comprimento normalizado das pétalas');

## Modelo e função de custo

O modelo é a reta $\hat{y} = w_0 + w_1 x$, implementada como `nn.Linear(1, 1)` ($w_1$ = `weight`, $w_0$ = `bias`). A função de custo é o MSE:

$$ J(W) = \frac{1}{M} \sum_{i=0}^{M-1} (\hat{y}_i - y_i)^2 $$

onde $M$ é o número de amostras do mini-batch (no mapa de custo, as 50 amostras). É o mesmo cálculo de `nn.MSELoss()`, sem o fator $\frac{1}{2}$. O esquema da otimização é o do diagrama abaixo (no diagrama, $N$ é o número de amostras, que chamamos de $M$, e $\mathbf{w}_k$ são os parâmetros no passo $k$, que aqui chamamos de $W_t$). Só o bloco "Gradiente Descendente" muda neste notebook: cada otimizador usa uma regra de atualização diferente.

![Otimização](https://raw.githubusercontent.com/vcasadei/images/master/RegressaoLinear_Otimizacao.png)

`compute_cost` calcula o custo $J$ para um ou vários vetores de parâmetros de uma vez (usado para desenhar o mapa de custo no plano $w_0 \times w_1$).

In [ ]:
def compute_cost(X_b, y, wT):
    '''
    Calcula o custo (MSE) da regressão linear.
    X_b: amostras com a coluna de 1s do bias, shape (M, 2)
    y:   valores corretos, shape (M, 1)
    wT:  vetor coluna de parâmetros (já transposto), shape (2, 1),
         ou (2, K) para avaliar K vetores de parâmetros de uma vez
    Retorna um array de shape (K,) com o custo de cada vetor de parâmetros.
    '''
    e = X_b.dot(wT) - y
    J = (e * e).mean(axis=0)
    return J

## Treinamento

`train_net` é a mesma função do notebook 2.1, com um argumento a mais, `optim`, que escolhe o otimizador. Todos os treinamentos partem do mesmo ponto, $w_0 = w_1 = 0$ (`load_state_dict` copia esses valores para os parâmetros da camada), e usam a mesma semente, portanto a mesma ordem dos mini-batches. A única diferença entre eles é a regra de atualização aplicada por `optimizer.step()`.

A função devolve o custo médio de cada época e o histórico dos parâmetros ao final de cada época.

In [ ]:
def make_optimizer(name, params, lr):
    # Hiperparâmetros fixos deste notebook: momentum=0.9, alpha=0.9, betas=(0.9, 0.99)
    if name == 'sgd':
        return torch.optim.SGD(params, lr=lr)
    if name == 'momentum':
        return torch.optim.SGD(params, lr=lr, momentum=0.9)
    if name == 'nesterov':
        return torch.optim.SGD(params, lr=lr, momentum=0.9, nesterov=True)
    if name == 'rmsprop':
        return torch.optim.RMSprop(params, lr=lr, alpha=0.9)
    if name == 'adam':
        return torch.optim.Adam(params, lr=lr, betas=(0.9, 0.99))
    raise ValueError(f'otimizador desconhecido: {name}')


def train_net(X, y, lr=0.5, n_epochs=50, batch_size=10, shuffle=True, rnd_seed=445, optim='sgd'):
    torch.manual_seed(rnd_seed)
    # DataLoader: entrega os dados em mini-batches de batch_size amostras, embaralhados a cada época
    dloader = DataLoader(TensorDataset(torch.from_numpy(X).float(),
                                       torch.from_numpy(y).float()),
                         batch_size=batch_size, shuffle=shuffle)
    # Modelo: uma camada linear com 1 entrada e 1 saída (w1 = weight, w0 = bias), iniciada com zeros
    model = torch.nn.Linear(1, 1)
    model.load_state_dict(dict(weight=torch.zeros(1, 1), bias=torch.zeros(1)))
    # Função de custo: MSE
    criterion = torch.nn.MSELoss()
    # Otimizador escolhido pelo argumento optim
    optimizer = make_optimizer(optim, model.parameters(), lr)

    losses = []
    w_history = np.zeros((n_epochs + 1, 2), np.float32)   # linha 0: parâmetros iniciais (zeros)
    model.train()
    # Laço de treinamento - épocas
    for epoch in range(n_epochs):
        cost = 0.0
        n_batches = 0
        for xb, yb in dloader:
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            optimizer.step()      # aplica a regra de atualização do otimizador escolhido

            cost += loss.item()   # .item(): converte o tensor de um elemento em número Python
            n_batches += 1

        losses.append(cost / n_batches)   # custo médio dos mini-batches da época
        w_history[epoch + 1, 0] = model.bias.detach()[0].item()        # .detach(): só leitura, sem gradiente
        w_history[epoch + 1, 1] = model.weight.detach()[0, 0].item()

    return losses, w_history

### Visualização

`show_plots` treina o modelo e desenha, época a época, três gráficos:

1. **Iris dataset**: os pontos (azul), a solução analítica (reta vermelha), a reta atual (preta), as predições (x pretos) e os erros de cada amostra (segmentos amarelos);
2. **W**: o mapa do custo $J$ no plano $w_0 \times w_1$ (azul = custo baixo, vermelho = custo alto), a solução analítica (x branco) e o caminho percorrido pelos parâmetros (pontos vermelhos, um por época);
3. **Custo**: o custo médio de cada época (linha) e o custo com os parâmetros ao final de cada época (pontos).

A solução analítica é a da equação normal $W^T = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$ (notebook 2.2). Ao final, a função imprime os parâmetros e o custo obtidos e os da solução analítica.

In [ ]:
def show_plots(x, y, lr=0.5, n_epochs=50, n_batch=50, shuffle=True, sleep=0.3, optim='sgd'):
    n_samples = y.shape[0]

    # minimização via gradiente descendente
    # -------------------------------------
    J_history, w_history = train_net(x, y, lr=lr, n_epochs=n_epochs, batch_size=n_batch,
                                     shuffle=shuffle, optim=optim)

    # valor ótimo, solução analítica
    # ------------------------------
    X_bias = np.hstack([np.ones((n_samples, 1)), x])
    w_opt = (np.linalg.inv((X_bias.T).dot(X_bias)).dot(X_bias.T)).dot(y)
    X_all = np.linspace(x.min(), x.max(), 100).reshape(100, 1)
    X_all_bias = np.hstack([np.ones((100, 1)), X_all])
    result_opt = X_all_bias.dot(w_opt)    # predição com os parâmetros ótimos

    # matriz bidimensional de parâmetros
    # ----------------------------------
    wmin = w_history.min(axis=0)
    wmax = w_history.max(axis=0)
    D = wmax - wmin
    wmin -= D
    wmax += D
    ww0, ww1 = np.meshgrid(np.linspace(wmin[0], wmax[0], 100), np.linspace(wmin[1], wmax[1], 100))
    w_grid = np.c_[ww0.ravel(), ww1.ravel()]
    J_grid = compute_cost(X_bias, y, w_grid.T)
    J_grid = J_grid.reshape(ww0.shape)

    # Gráficos
    # --------
    fig = plt.figure(figsize=(18, 6))
    fig.suptitle(f'{optim}: lr={lr}, n_epochs={n_epochs}, n_batch={n_batch}', fontsize=15)
    ax_line = fig.add_subplot(1, 3, 1)    # reta
    ax_grid = fig.add_subplot(1, 3, 2)    # espaço de parâmetros
    ax_loss = fig.add_subplot(1, 3, 3)    # função de custo

    ax_loss.plot(J_history)
    ax_loss.set_title('Custo', fontsize=15)
    ax_loss.set_xlabel('épocas', fontsize=10)
    ax_loss.set_ylabel('MSE', fontsize=10)

    ax_line.scatter(x, y, marker='o', c='b')    # pontos do Iris dataset
    ax_line.set_title('Iris dataset', fontsize=15)
    ax_line.set_xlabel('x', fontsize=10)
    ax_line.set_ylabel('y', fontsize=10)
    ax_line.plot(X_all, result_opt, c='r')      # solução analítica: reta vermelha

    ax_grid.pcolormesh(ww0, ww1, J_grid, cmap=plt.cm.coolwarm)
    ax_grid.contour(ww0, ww1, J_grid, 20)
    ax_grid.scatter(w_opt[0], w_opt[1], marker='x', c='w')   # solução analítica
    ax_grid.set_title('W', fontsize=15)
    ax_grid.set_xlabel('w0', fontsize=10)
    ax_grid.set_ylabel('w1', fontsize=10)

    # Plot dinâmico: um quadro por época
    # ----------------------------------
    ln, sc, ln_v = None, None, []
    for i, w in enumerate(w_history):
        if ln: ln.remove()
        if sc: sc.remove()
        for lnv in ln_v:
            lnv.remove()

        y_hat = X_all_bias.dot(w.T)    # reta obtida pelo gradiente descendente
        w = w.reshape(1, 2)
        J = compute_cost(X_bias, y, w.T)

        ax_loss.scatter(i, J[0], c='k', s=15)

        ln, = ax_line.plot(X_all, y_hat, c='k', lw=2)    # reta preta

        y_pred = X_bias.dot(w.T)
        sc = ax_line.scatter(x, y_pred, c='k', marker='x')
        ln_v = [ax_line.plot([x[k, 0], x[k, 0]], [y_pred[k, 0], y[k, 0]], c='y')[0]
                for k in range(n_samples)]

        ax_grid.scatter(w_history[i, 0], w_history[i, 1], c='r', marker='o')

        display.display(fig)
        display.clear_output(wait=True)
        time.sleep(sleep)
    plt.close(fig)
    display.display(fig)
    print(f'{"w final (" + optim + "):":<29}w0 = {w_history[-1, 0]:.4f}, w1 = {w_history[-1, 1]:.4f}, '
          f'custo = {compute_cost(X_bias, y, w_history[-1:].T)[0]:.5f}')
    print(f'{"w ótimo (solução analítica):":<29}w0 = {w_opt[0, 0]:.4f}, w1 = {w_opt[1, 0]:.4f}, '
          f'custo = {compute_cost(X_bias, y, w_opt)[0]:.5f}')

## Comparação dos otimizadores

Os cinco experimentos a seguir usam os mesmos hiperparâmetros: `lr=0.05`, `n_epochs=40` e `n_batch=50`. Como o conjunto tem 50 amostras, `n_batch=50` significa um único mini-batch por época: cada época é **um** passo, calculado com todas as amostras (gradiente descendente em lote). Assim, as diferenças entre os caminhos vêm só da regra de atualização, não do ruído dos mini-batches.

Antes de executar, olhe as equações e preveja: qual otimizador deve ultrapassar o mínimo e voltar? Qual deve dar passos de tamanho quase constante?

### SGD

In [ ]:
show_plots(x, y, lr=0.05, n_epochs=40, n_batch=50, sleep=0.1, optim='sgd')

### SGD com momentum

In [ ]:
show_plots(x, y, lr=0.05, n_epochs=40, n_batch=50, sleep=0.1, optim='momentum')

### SGD com momentum de Nesterov

In [ ]:
show_plots(x, y, lr=0.05, n_epochs=40, n_batch=50, sleep=0.1, optim='nesterov')

### RMSprop

In [ ]:
show_plots(x, y, lr=0.05, n_epochs=40, n_batch=50, sleep=0.1, optim='rmsprop')

### Adam

In [ ]:
show_plots(x, y, lr=0.05, n_epochs=40, n_batch=50, sleep=0.1, optim='adam')

### Resumo da comparação

A célula abaixo treina os cinco otimizadores de novo, sem animação, e mostra num só gráfico os caminhos no plano $w_0 \times w_1$ e as curvas de custo. Os números valem só para esta configuração (`lr=0.05`, 40 épocas, lote completo).

In [ ]:
X_bias = np.hstack([np.ones((x.shape[0], 1)), x])
w_opt = np.linalg.inv(X_bias.T @ X_bias) @ X_bias.T @ y
J_opt = compute_cost(X_bias, y, w_opt)[0]

optim_names = ['sgd', 'momentum', 'nesterov', 'rmsprop', 'adam']
fig, (ax_w, ax_j) = plt.subplots(1, 2, figsize=(14, 5))
print(f'{"otimizador":<10} {"w0":>8} {"w1":>8} {"custo final":>12}')
for name in optim_names:
    _, w_hist = train_net(x, y, lr=0.05, n_epochs=40, batch_size=50, optim=name)
    J_hist = compute_cost(X_bias, y, w_hist.T)       # custo com os parâmetros ao final de cada época
    ax_w.plot(w_hist[:, 0], w_hist[:, 1], 'o-', ms=3, label=name)
    ax_j.plot(J_hist, label=name)
    print(f'{name:<10} {w_hist[-1, 0]:8.4f} {w_hist[-1, 1]:8.4f} {J_hist[-1]:12.5f}')
print(f'{"analítica":<10} {w_opt[0, 0]:8.4f} {w_opt[1, 0]:8.4f} {J_opt:12.5f}')

ax_w.scatter(w_opt[0], w_opt[1], marker='x', c='k', s=100, label='solução analítica')
ax_w.set_xlabel('w0'); ax_w.set_ylabel('w1'); ax_w.set_title('Caminho dos parâmetros'); ax_w.legend()
ax_j.axhline(J_opt, c='k', ls='--', lw=1, label='custo mínimo')
ax_j.set_yscale('log'); ax_j.set_xlabel('épocas'); ax_j.set_ylabel('MSE (escala log)')
ax_j.set_title('Custo'); ax_j.legend();

## Exercícios

1. Para cada otimizador, procure valores de taxa de aprendizado (`lr`) e de tamanho do mini-batch (`n_batch`) que levem os parâmetros para perto da solução analítica nas mesmas 40 épocas. Use `train_net` e `compute_cost` (sem animação) para testar várias combinações de uma vez, por exemplo `lr` em `[0.01, 0.05, 0.1, 0.5, 1.0]` e `n_batch` em `[50, 10]`, e imprima o custo final de cada uma. Depois use `show_plots` para ver o caminho da melhor combinação de um dos otimizadores.

In [ ]:
# TODO: escreva sua solução aqui


In [ ]:
# TODO: escreva sua solução aqui


*Escreva sua resposta aqui.*

2. Com `n_batch=50`, o "SGD" deste notebook é estocástico? Quantas atualizações dos parâmetros acontecem em 40 épocas com `n_batch=50` e com `n_batch=10`?

*Escreva sua resposta aqui.*

3. Pela equação do RMSprop, explique por que ele não converge com `lr=0.5`, enquanto o SGD com o mesmo `lr` chega perto do mínimo. O que acontece com o passo do SGD quando os parâmetros se aproximam do mínimo? E com o passo do RMSprop?

*Escreva sua resposta aqui.*

4. Quais são as principais conclusões que podemos tirar deste experimento?

*Escreva sua resposta aqui.*

## Aprendizados

- Todos os otimizadores vistos usam o mesmo gradiente $g_t$ calculado por `loss.backward()`; o que muda é a regra de `optimizer.step()`.
- O momentum (e o Nesterov) acumula os gradientes anteriores numa velocidade $v$; RMSprop e Adam dividem o passo de cada parâmetro pela raiz de uma média móvel dos gradientes ao quadrado, e o Adam também usa uma média móvel dos gradientes.
- O passo do SGD encolhe perto do mínimo; o do RMSprop e do Adam fica da ordem de $\eta$, o que muda a faixa boa de taxa de aprendizado de cada método.
- Comparar otimizadores exige o mesmo ponto inicial, os mesmos dados e várias taxas de aprendizado; a conclusão vale para o problema e os hiperparâmetros testados.